# StegoCrypt

Authenticated AES-GCM encryption with LSB image steganography in Python.


### Overview
This notebook demonstrates hiding encrypted secret messages and files inside PNG images:
* Secrets are encrypted with AES-256-GCM before embedding so bits look like random camera sensor noise
* Bits are injected into the least significant bit (LSB) of each pixel color channel
* An authentication tag verifies carrier integrity and blocks tampered or cropped images


### Setup
Install Pillow and Cryptography if running in a new environment or Google Colab.


In [ ]:
%pip install pillow cryptography


In [ ]:
import os
import time
import struct
from PIL import Image
from cryptography.exceptions import InvalidTag
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
from cryptography.hazmat.primitives import hashes

MAGIC = b"STG1"
HEADER_LEN = 36  # [4B magic] [12B nonce] [16B tag] [4B ct_len]
SALT = b"stego_fixed_salt_v1"  # TODO: store random per-image salt in header if migrating to multi-tenant


### Steganography Functions
Key derivation, capacity estimation, and LSB embedding and extraction routines.


In [ ]:
def derive_key(passphrase: str) -> bytes:
    kdf = PBKDF2HMAC(
        algorithm=hashes.SHA256(),
        length=32,
        salt=SALT,
        iterations=100_000,
    )
    return kdf.derive(passphrase.encode("utf-8"))

def get_capacity(img: Image.Image) -> int:
    # 3 bits per pixel (R, G, B). Alpha is left untouched to prevent transparency speckles.
    total_bits = img.width * img.height * 3
    total_bytes = total_bits // 8
    return max(0, total_bytes - HEADER_LEN)

def embed_bytes(img: Image.Image, data: bytes, key: bytes) -> Image.Image:
    # TODO: support palette P-mode images directly without converting
    if img.mode not in ("RGB", "RGBA"):
        img = img.convert("RGB")

    nonce = os.urandom(12)
    cipher = AESGCM(key)
    raw = cipher.encrypt(nonce, data, MAGIC)
    ct, tag = raw[:-16], raw[-16:]

    hdr = struct.pack(">4s12s16sI", MAGIC, nonce, tag, len(ct))
    packet = hdr + ct

    total_bits = len(packet) * 8
    max_bits = img.width * img.height * 3
    if total_bits > max_bits:
        raise ValueError(f"Payload too large ({len(packet)}B). Carrier max is {max_bits // 8}B")

    bits = [(b >> s) & 1 for b in packet for s in (7, 6, 5, 4, 3, 2, 1, 0)]
    pixels = bytearray(img.tobytes())

    if img.mode == "RGBA":
        bit_idx = 0
        for i in range(0, len(pixels), 4):
            for c in range(3):
                if bit_idx < len(bits):
                    pixels[i + c] = (pixels[i + c] & ~1) | bits[bit_idx]
                    bit_idx += 1
                else:
                    break
            if bit_idx >= len(bits):
                break
    else:
        for i in range(len(bits)):
            pixels[i] = (pixels[i] & ~1) | bits[i]

    return Image.frombytes(img.mode, img.size, bytes(pixels))

def extract_bytes(img: Image.Image, key: bytes) -> bytes:
    if img.mode not in ("RGB", "RGBA"):
        img = img.convert("RGB")

    pixels = img.tobytes()
    mode = img.mode

    def channels():
        if mode == "RGBA":
            for i in range(0, len(pixels), 4):
                yield pixels[i]
                yield pixels[i + 1]
                yield pixels[i + 2]
        else:
            for b in pixels:
                yield b

    def read_n(gen, n):
        out = bytearray()
        for _ in range(n):
            val = 0
            for shift in (7, 6, 5, 4, 3, 2, 1, 0):
                try:
                    ch = next(gen)
                except StopIteration:
                    return None
                val |= (ch & 1) << shift
            out.append(val)
        return bytes(out)

    gen = channels()
    hdr = read_n(gen, HEADER_LEN)
    if not hdr:
        return None

    magic, nonce, tag, ct_len = struct.unpack(">4s12s16sI", hdr)
    if magic != MAGIC:
        return None

    cap = get_capacity(img)
    if ct_len > cap:
        return None

    ct = read_n(gen, ct_len)
    if not ct or len(ct) != ct_len:
        return None

    cipher = AESGCM(key)
    try:
        return cipher.decrypt(nonce, ct + tag, MAGIC)
    except InvalidTag:
        return None


### Demonstration: Hide & Reveal Secret
Creates a sample 300x300 carrier image, embeds confidential text, and extracts it using password-derived encryption.


### Upload Your Own Image
Run this cell to upload a carrier image from your computer. PNG or BMP works best. JPEG is lossy and will destroy the hidden bits.

In [ ]:
import os, io
from google.colab import files

print("Upload a carrier image (PNG or BMP):")
uploaded = files.upload()
img_filename = list(uploaded.keys())[0]
carrier = Image.open(io.BytesIO(uploaded[img_filename]))
print(f"Loaded: {img_filename} ({carrier.width}x{carrier.height} {carrier.mode})")
print(f"Max payload capacity: {get_capacity(carrier):,} bytes (~{get_capacity(carrier) // 1024} KB)")

### Embed a Secret and Download the Stego Image
Type your secret message below, pick a password, then run to get the stego PNG back.

In [ ]:
secret_text = "your secret message here"
password = "your_password_here"

key = derive_key(password)
stego = embed_bytes(carrier, secret_text.encode("utf-8"), key)

out_name = "stego_" + img_filename.rsplit(".", 1)[0] + ".png"
stego.save(out_name, format="PNG")
print(f"Embedded {len(secret_text)} chars into {out_name}")
files.download(out_name)

### Extract a Secret from a Stego Image
Upload a stego PNG you made earlier, enter the password, and get the secret back.

In [ ]:
print("Upload a stego image:")
stego_up = files.upload()
stego_name = list(stego_up.keys())[0]
stego_img = Image.open(io.BytesIO(stego_up[stego_name]))

password = "your_password_here"
key = derive_key(password)
recovered = extract_bytes(stego_img, key)

if recovered is None:
    print("Nothing found. Wrong password or image was modified.")
else:
    try:
        print(f"Secret: {recovered.decode('utf-8')}")
    except UnicodeDecodeError:
        out = "extracted_secret.bin"
        with open(out, "wb") as f:
            f.write(recovered)
        print(f"Binary payload ({len(recovered)} bytes). Downloading...")
        files.download(out)

In [ ]:
carrier = Image.new("RGB", (300, 300), color=(52, 101, 164))
cap = get_capacity(carrier)
print(f"Carrier capacity: {cap:,} bytes (~{cap // 1024} KB)")

password = "super_secure_vault_passphrase"
key = derive_key(password)
secret_note = "Confidential rendezvous: 40.7128N, 74.0060W at 21:00 UTC."

stego_carrier = embed_bytes(carrier, secret_note.encode("utf-8"), key)
recovered_bytes = extract_bytes(stego_carrier, key)

assert recovered_bytes == secret_note.encode("utf-8")
print(f"Original:  {secret_note}")
print(f"Extracted: {recovered_bytes.decode('utf-8')}")
print("Status: Verified successfully")


### Security Tests
Verifies carrier pixel tamper detection, wrong password rejection, and capacity overflow limits.


In [ ]:
img = Image.new("RGB", (200, 200), color=(70, 130, 180))
k = os.urandom(32)
msg = b"Target payload data"

stego_img = embed_bytes(img, msg, k)
assert extract_bytes(stego_img, k) == msg
print("Embed and extraction baseline: PASS")

tampered_bytes = bytearray(stego_img.tobytes())
tampered_bytes[100] ^= 0x01
tampered_img = Image.frombytes(stego_img.mode, stego_img.size, bytes(tampered_bytes))
assert extract_bytes(tampered_img, k) is None
print("Tampered carrier pixel rejection: PASS")

bad_key = os.urandom(32)
assert extract_bytes(stego_img, bad_key) is None
print("Wrong key rejected: PASS")

clean_img = Image.new("RGB", (100, 100), color=(255, 0, 0))
assert extract_bytes(clean_img, k) is None
print("Clean carrier returns None: PASS")

tiny_img = Image.new("RGB", (10, 10))
try:
    embed_bytes(tiny_img, b"X" * 1000, k)
    assert False
except ValueError:
    print("Capacity overflow guard: PASS")


### Speed Benchmark
Measures embedding and extraction speed across 30 rounds on a 500x500 image.


In [ ]:
img = Image.new("RGB", (500, 500), color=(30, 30, 30))
key = os.urandom(32)
cap = get_capacity(img)
print(f"Carrier: 500x500 RGB | Usable payload capacity: {cap:,} bytes (~{cap // 1024} KB)")

test_payload = os.urandom(16 * 1024)
rounds = 30

t0 = time.time()
for _ in range(rounds):
    stego_img = embed_bytes(img, test_payload, key)
embed_time = time.time() - t0

t1 = time.time()
for _ in range(rounds):
    res = extract_bytes(stego_img, key)
extract_time = time.time() - t1

total_mb = (len(test_payload) * rounds) / (1024 * 1024)
print(f"Payload: 16 KB across {rounds} runs")
print(f"Embed speed:   {total_mb / embed_time:.2f} MB/s ({embed_time / rounds * 1000:.2f} ms/embed)")
print(f"Extract speed: {total_mb / extract_time:.2f} MB/s ({extract_time / rounds * 1000:.2f} ms/extract)")
